# Практическая работа: Vanilla RAG для корпоративной базы знаний

Загрузка данных

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Find the project when the kernel starts in its root or notebooks folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data/raw/articles.json").is_file()
     and (path / "configs/prompts.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the project folder before running this notebook.")

DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "configs"


In [ ]:
def load_json(relative_path):
    with (DATA_DIR / relative_path).open("r", encoding="utf-8") as file:
        return json.load(file)


In [ ]:
articles = load_json("raw/articles.json")
questions = load_json("evaluation/questions.json")
ground_truth = load_json("evaluation/ground_truth.json")


In [ ]:
print(type(articles), len(articles))
print(type(questions), len(questions))
print(type(ground_truth), len(ground_truth))

In [ ]:
print(articles[0])
print(questions[0])
print(ground_truth[0])

In [ ]:
def merge_lists_by_id(list1,list2):
  dict1 = {item['id']: item for item in list1}

  for item in list2:
    dict1[item['id']].update(item)

  return list(dict1.values())

golden_set = merge_lists_by_id(questions,ground_truth)

In [ ]:
golden_set

In [ ]:
#!pip install groq -q

In [ ]:
from getpass import getpass
from groq import Groq

api_key = getpass("GROQ_API_KEY: ")

client = Groq(
    api_key=api_key
)


chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
             "content": f"""Ответь кратко и по существу на вопрос.
            Не придумывай дополнительные детали.
            Если точного ответа не знаешь — так и скажи.

            Вопрос: За сколько рабочих дней до поездки нужно подать заявку на командировку в TravelHub?"""
    }
    ],
    model="openai/gpt-oss-120b"
)

print(chat_completion.choices[0].message.content)

In [ ]:
import yaml

with (CONFIG_DIR / "prompts.yaml").open("r", encoding="utf-8") as f:
    prompts = yaml.safe_load(f)

SYSTEM_PROMPT = prompts["ask_groq"]["system"]

In [ ]:
def ask_groq(question):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ],
        model="openai/gpt-oss-120b"
    )

    return chat_completion.choices[0].message.content

In [ ]:
for item in golden_set:
  question = item['question']
  answer = ask_groq(question)

  item['groq_answer'] = answer

In [ ]:
golden_set[0]

Посчитаем качетсво ответов с помощью метрик BLEU и ROUGE

In [ ]:
#!pip install evaluate sacrebleu rouge_score

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

In [ ]:
references = [item["answer"] for item in golden_set]
predictions = [item["groq_answer"] for item in golden_set]

In [ ]:
bleu_result = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print(bleu_result)

In [ ]:
rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

print(rouge_result)

Zero-shot модель без доступа к корпоративной базе показывает низкое совпадение с эталонными ответами. BLEU составил 1.62, ROUGE-1 — 0.144, ROUGE-2 — 0.052, ROUGE-L — 0.139. Это ожидаемо, поскольку вопросы относятся к внутренним регламентам компании, которых модель не знает. Дополнительно модель генерирует примерно в два раза более длинные ответы, чем эталон.

In [ ]:
def judge_answer(question, prediction, reference):
  prompt = f"""
  Оцени ответ модели на вопрос.

  Вопрос:
  {question}

  Эталонный ответ:
  {reference}

  Ответ модели:
  {prediction}

  Поставь оценку:
  0 — ответ неверный;
  1 — ответ частично верный;
  2 — ответ полностью верный.

  Верни только число: 0, 1 или 2.
  """

  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": prompt
          }
      ],
      model="openai/gpt-oss-120b"
  )

  return chat_completion.choices[0].message.content

In [ ]:
judge_answer(
    golden_set[0]["question"],
    golden_set[0]["groq_answer"],
    golden_set[0]["answer"]
)

In [ ]:
for item in golden_set:
  score = judge_answer(
      item['question'],
      item['groq_answer'],
      item['answer']
  )

  item['judge_score'] = int(score)

In [ ]:
golden_set[0]

In [ ]:
#!pip install vllm -q

In [ ]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
llm = LLM(
    model="Qwen/Qwen2.5-0.5B-Instruct"
)

In [ ]:
sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=40
)

In [ ]:
prompt =  """Ответь кратко и по существу на вопрос.
Не придумывай дополнительные детали.
Если точного ответа не знаешь — так и скажи.

Вопрос: За сколько рабочих дней до поездки нужно подать заявку на командировку в TravelHub?
"""

outputs = llm.generate(
    [prompt],
    sampling_params
)

print(outputs[0].outputs[0].text)

In [ ]:
def ask_local(question):
    prompt = f"""Ответь кратко и по существу на вопрос.
    Не придумывай дополнительные детали.
    Если точного ответа не знаешь — так и скажи.

    Вопрос: {question}
    """

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]

    outputs = llm.chat(
        messages,
        sampling_params=sampling_params
    )

    return outputs[0].outputs[0].text

In [ ]:
ask_local(golden_set[0]["question"])

In [ ]:
for item in golden_set:
    item["local_answer"] = ask_local(item["question"])

In [ ]:
local_predictions = [item["local_answer"] for item in golden_set]

In [ ]:
local_bleu = bleu.compute(
    predictions=local_predictions,
    references=[[ref] for ref in references]
)

In [ ]:
local_bleu

In [ ]:
local_rouge = rouge.compute(
    predictions=local_predictions,
    references=references
)

In [ ]:
local_rouge

In [ ]:
for item in golden_set:
  score = judge_answer(
      item['question'],
      item['local_answer'],
      item['answer']
  )

  item['local_judge_score'] = int(score)

In [ ]:
groq_mean = sum(item["judge_score"] for item in golden_set) / len(golden_set)


local_mean = sum(item["local_judge_score"] for item in golden_set) / len(golden_set)

print("Groq mean judge =",groq_mean)
print("Local mean judge =",local_mean)


Обе zero-shot модели показали крайне низкое качество на вопросах внутренней корпоративной базы. Local Qwen получил немного более высокие BLEU/ROUGE из-за более коротких и лексически похожих ответов, однако LLM-as-a-judge показал почти одинаково низкую фактическую корректность: 0.08 для Groq и 0.06 для локальной модели. Это подтверждает, что без доступа к корпоративным документам модели не способны надёжно отвечать на такие вопросы.

In [ ]:
articles

# RAG

In [ ]:
from langchain_core.documents import Document

In [ ]:
documents = []

for item in articles:
    doc = Document(
        page_content=f"{item['title']}\n{item["text"]}",
        metadata={
            "id": item["id"],
            "title": item["title"]
        }
    )

    documents.append(doc)

In [ ]:
documents[0]

In [ ]:
for doc in documents:
    print(len(doc.page_content), doc.metadata["title"])

Так как у нас документы короткие, примерно 800 символов, то лучше брать 1 чанк за 1 документ

In [ ]:
chunks = documents

In [ ]:
#!pip install langchain_huggingface -q

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_kwargs = {'device': 'cuda'}
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
encode_kwargs = {'normalize_embeddings': True}


embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [ ]:
vectors = embeddings.embed_documents(
    ["Привет, мир!", "Как делать embeddings?"]
)

In [ ]:
print(type(vectors))
print(len(vectors))
print(len(vectors[0]))

In [ ]:
#pip install langchain-community faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

In [ ]:
query = "За сколько рабочих дней нужно подать заявку на командировку?"

results = vectorstore.similarity_search(
    query,
    k=2
)

In [ ]:
for doc in results:
    print(doc.metadata)
    print(doc.page_content)
    print()

In [ ]:
for item in golden_set:
    query = item["question"]
    results = vectorstore.similarity_search(query, k=2)
    item['retrieved_ids'] = [doc.metadata['id'] for doc in results]

In [ ]:
golden_set[0]["retrieved_ids"]

In [ ]:
def get_expected_article_id(question_id):
  q_num = int(question_id.replace("q",""))

  if 1 <= q_num <= 7:
        return "1"
  elif 8 <= q_num <= 13:
        return "2"
  elif 14 <= q_num <= 19:
        return "3"
  elif 20 <= q_num <= 25:
        return "4"
  elif 26 <= q_num <= 30:
        return "5"
  elif 31 <= q_num <= 36:
        return "6"
  elif 37 <= q_num <= 41:
        return "7"
  elif 42 <= q_num <= 45:
        return "8"
  elif 46 <= q_num <= 48:
        return "9"
  elif 49 <= q_num <= 50:
        return "10"

In [ ]:
for item in golden_set:
    item["expected_article_id"] = get_expected_article_id(item["id"])

In [ ]:
def recall_at_k(golden_set, k):
    correct = 0

    for item in golden_set:
        if item["expected_article_id"] in item["retrieved_ids"][:k]:
            correct += 1

    return correct / len(golden_set)

In [ ]:
print(recall_at_k(golden_set, 1))
print(recall_at_k(golden_set, 2))

In [ ]:
results

In [ ]:
context = "\n\n".join(
    doc.page_content for doc in results
)

In [ ]:
print(context)

In [ ]:
def ask_rag(question):
    results = vectorstore.similarity_search(question, k=2)

    context = "\n\n".join(
        doc.page_content for doc in results
    )

    prompt = f"""Ответь на вопрос, используя только контекст ниже.
    Если в контексте нет ответа, скажи: "Не знаю".

    Контекст:
    {context}

    Вопрос:
    {question}

    Ответь кратко и по существу.
    """

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="openai/gpt-oss-120b"
    )

    return chat_completion.choices[0].message.content

In [ ]:
ask_rag(golden_set[0]["question"])

In [ ]:
for item in golden_set:
    question = item["question"]
    rag_answer = ask_rag(question)
    item['rag_answer'] = rag_answer

In [ ]:
rag_predictions = [item["rag_answer"] for item in golden_set]
references = [item["answer"] for item in golden_set]

In [ ]:
rag_bleu = bleu.compute(
    predictions=rag_predictions,
    references=[[ref] for ref in references]
)

In [ ]:
rag_rouge = rouge.compute(
    predictions=rag_predictions,
    references=references
)

In [ ]:
for item in golden_set:
    score = judge_answer(
        item["question"],
        item["answer"],
        item["rag_answer"]
    )

    item["rag_judge_score"] = int(score)

In [ ]:
rag_mean = sum(
    item["rag_judge_score"] for item in golden_set
) / len(golden_set)

In [ ]:
rag_bleu

In [ ]:
rag_rouge

In [ ]:
rag_mean

Основной прирост качества связан не с заменой LLM, а с добавлением релевантного корпоративного контекста через RAG. При этом качество retrieval критично: после улучшения представления документов через title + text Recall@2 вырос до 0.92, что напрямую повысило итоговое качество генерации.
Zero-shot LLM почти бесполезна для закрытой корпоративной базы.
RAG резко улучшает factual accuracy, но итоговое качество ограничено качеством retrieval.

| Модель / режим | BLEU | ROUGE-1 | ROUGE-2 | ROUGE-L | LLM-as-a-judge |
|---|---:|---:|---:|---:|---:|
| Groq zero-shot | 1.62 | 0.144 | 0.052 | 0.139 | 0.08 |
| Local Qwen2.5-0.5B zero-shot | 3.83 | 0.158 | 0.044 | 0.153 | 0.06 |
| Groq + RAG | **30.47** | **0.767** | **0.301** | **0.759** | **1.74** |